In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from ignite.metrics.rec_sys import NDCG

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn


In [2]:
import gzip
import ast


In [3]:
from pathlib import Path

def load_data(path):
    data = []

    with gzip.open(path, "rt", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                data.append(ast.literal_eval(line))

    return pd.DataFrame(data)

In [4]:
# Путь к папке проекта, если ноутбук лежит в папке notebooks/
project_dir = Path("..")

processed_dir = project_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

reviews_path = processed_dir / "reviews.parquet"
users_items_path = processed_dir / "users_items.parquet"

if reviews_path.exists() and users_items_path.exists():
    reviews = pd.read_parquet(reviews_path)
    users_items = pd.read_parquet(users_items_path)
    print("Загрузили готовые датафреймы")
else:
    reviews = load_data(project_dir / "data" / "raw" / "australian_user_reviews.json.gz")
    users_items = load_data(project_dir / "data" / "raw" / "australian_users_items.json.gz")

    reviews.to_parquet(reviews_path, index=False)
    users_items.to_parquet(users_items_path, index=False)
    print("Загрузили исходные файлы и сохранили датафреймы")

Загрузили исходные файлы и сохранили датафреймы


In [6]:
reviews.head()

,user_id,user_url,reviews
0,76561197970982479,http://steamcommunity.com/profiles/76561197970...,"[{'funny': '', 'posted': 'Posted November 5, 2..."
1,js41637,http://steamcommunity.com/id/js41637,"[{'funny': '', 'posted': 'Posted June 24, 2014..."
2,evcentric,http://steamcommunity.com/id/evcentric,"[{'funny': '', 'posted': 'Posted February 3.',..."
3,doctr,http://steamcommunity.com/id/doctr,"[{'funny': '', 'posted': 'Posted October 14, 2..."
4,maplemage,http://steamcommunity.com/id/maplemage,"[{'funny': '3 people found this review funny',..."


In [7]:
users_items.head()

,user_id,items_count,steam_id,user_url,items
0,76561197970982479,277,76561197970982479,http://steamcommunity.com/profiles/76561197970...,"[{'item_id': '10', 'item_name': 'Counter-Strik..."
1,js41637,888,76561198035864385,http://steamcommunity.com/id/js41637,"[{'item_id': '10', 'item_name': 'Counter-Strik..."
2,evcentric,137,76561198007712555,http://steamcommunity.com/id/evcentric,"[{'item_id': '1200', 'item_name': 'Red Orchest..."
3,Riot-Punch,328,76561197963445855,http://steamcommunity.com/id/Riot-Punch,"[{'item_id': '10', 'item_name': 'Counter-Strik..."
4,doctr,541,76561198002099482,http://steamcommunity.com/id/doctr,"[{'item_id': '300', 'item_name': 'Day of Defea..."
